# UrbanRisk AI — M1: Chennai AQI Forecasting (Baseline)
**Milestone 1 of 9** — Data acquisition → cleaning → features → baseline → XGBoost → evaluation → SHAP

This notebook builds the AQI forecasting pipeline for **+6h / +12h / +24h** ahead predictions, for a single
representative Chennai point to start. Later milestones extend this to the full 20–50 zone grid.

### Data sources used here (verified, free, no paid key required)
| Source | What it gives | Why |
|---|---|---|
| **Open-Meteo Historical Weather Archive** (`archive-api.open-meteo.com`) | Hourly temperature, humidity, pressure, rainfall, wind — back to 1940 | Free, keyless, gap-free reanalysis (ERA5), reliable for bulk historical pulls |
| **Open-Meteo Air Quality API** (`air-quality-api.open-meteo.com`) | Hourly PM2.5, PM10, NO2, SO2, O3, and a European/US AQI — supports historical `start_date`/`end_date` | Free, keyless, CAMS-based **model reanalysis**, good for bulk historical backfill |
| **WAQI / aqicn.org (CPCB ground stations)** | Real, ground-truth CPCB readings for Chennai stations (Manali, Alandur, Arumbakkam, Kodungaiyur, Perungudi, Royapuram, Velachery, etc.) | This is the *authoritative* ground-truth source, but bulk historical access needs a free token (`aqicn.org/data-platform/token/`) and institutional history downloads can be slow to arrange |

### Honest note on data strategy
CPCB's own historical bulk-download portal is not reliably scriptable, and WAQI's public historical API access
beyond "current + last 2 days" is granted case-by-case after registering for a token. So the pragmatic path for
M1 is:
1. Use **Open-Meteo's Air Quality archive** as the historical AQI/pollutant backbone (clearly a model-reanalysis
   product, not raw station readings — we treat it as *directionally correct*, not ground truth).
2. Register a free WAQI token (`aqicn.org/data-platform/token/`) and, from **today onward**, start logging the
   real CPCB station feed for Chennai — this becomes your ground-truth stream and lets you later calibrate /
   validate the reanalysis-trained model, and swap it in once enough history accumulates.
3. Do not fabricate or assume CPCB numbers — every value in this notebook comes from a live API call, and every
   cell prints its source and row counts so you can sanity-check it.

**Action for you before running:** nothing required for Cells 1–9 (fully keyless). If you later want the ground-truth
WAQI stream (recommended for M1 validation and required from M2 onward for the flood/incident side), get a free
token at `https://aqicn.org/data-platform/token/` and paste it into the `WAQI_TOKEN` variable in the optional cell
near the end.


## 1. Setup

In [ ]:
!pip install -q xgboost shap scikit-learn pandas numpy requests matplotlib

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (12, 4)

# Chennai reference point (central Chennai). We start with one point for M1;
# M2+ will replicate this pull across a zone grid (lat/lon list).
CHENNAI_LAT = 13.0827
CHENNAI_LON = 80.2707

# Historical window: Open-Meteo archive has a short ingestion lag (~a few days),
# so we pull up to 5 days before "today" to stay safely inside published data.
END_DATE = (datetime.utcnow() - timedelta(days=5)).strftime("%Y-%m-%d")
START_DATE = (datetime.utcnow() - timedelta(days=5) - timedelta(days=730)).strftime("%Y-%m-%d")  # ~2 years
print("Pulling data from", START_DATE, "to", END_DATE)


## 2. Load data — Historical Weather (Open-Meteo Archive)
Endpoint: `https://archive-api.open-meteo.com/v1/archive`. Keyless, hourly, ERA5-based, gap-free.


In [ ]:
def fetch_weather_history(lat, lon, start_date, end_date):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join([
            "temperature_2m",
            "relative_humidity_2m",
            "pressure_msl",
            "precipitation",
            "rain",
            "wind_speed_10m",
            "wind_direction_10m",
        ]),
        "timezone": "Asia/Kolkata",
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()
    df = pd.DataFrame(data["hourly"])
    df["time"] = pd.to_datetime(df["time"])
    df = df.rename(columns={
        "time": "timestamp",
        "temperature_2m": "temperature",
        "relative_humidity_2m": "humidity",
        "pressure_msl": "pressure",
        "precipitation": "rainfall_1h",
        "wind_speed_10m": "wind_speed",
        "wind_direction_10m": "wind_direction",
    })
    return df

weather_df = fetch_weather_history(CHENNAI_LAT, CHENNAI_LON, START_DATE, END_DATE)
print("Weather rows:", len(weather_df))
weather_df.head()


## 3. Load data — Historical Air Quality (Open-Meteo Air Quality API)
Endpoint: `https://air-quality-api.open-meteo.com/v1/air-quality`. Keyless, hourly, CAMS-reanalysis based.
This is our AQI/pollutant backbone for M1 — flagged throughout as `source = "openmeteo_reanalysis"` so it's
never confused with a real CPCB ground reading.


In [ ]:
def fetch_air_quality_history(lat, lon, start_date, end_date):
    url = "https://air-quality-api.open-meteo.com/v1/air-quality"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join([
            "pm2_5",
            "pm10",
            "nitrogen_dioxide",
            "sulphur_dioxide",
            "ozone",
            "us_aqi",
        ]),
        "timezone": "Asia/Kolkata",
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()
    df = pd.DataFrame(data["hourly"])
    df["time"] = pd.to_datetime(df["time"])
    df = df.rename(columns={
        "time": "timestamp",
        "pm2_5": "PM2_5",
        "pm10": "PM10",
        "nitrogen_dioxide": "NO2",
        "sulphur_dioxide": "SO2",
        "ozone": "O3",
        "us_aqi": "AQI",
    })
    df["aqi_source"] = "openmeteo_reanalysis"
    return df

aq_df = fetch_air_quality_history(CHENNAI_LAT, CHENNAI_LON, START_DATE, END_DATE)
print("Air quality rows:", len(aq_df))
aq_df.head()


## 4. Inspect the data — schema, ranges, missingness

In [ ]:
print("=== Weather ===")
print(weather_df.dtypes)
print(weather_df.describe())
print("\nMissing values per column:")
print(weather_df.isna().sum())

print("\n=== Air Quality ===")
print(aq_df.dtypes)
print(aq_df.describe())
print("\nMissing values per column:")
print(aq_df.isna().sum())


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(weather_df["timestamp"], weather_df["temperature"], linewidth=0.6)
axes[0].set_title("Temperature (raw)")
axes[1].plot(aq_df["timestamp"], aq_df["AQI"], linewidth=0.6, color="firebrick")
axes[1].set_title("AQI (raw, Open-Meteo reanalysis)")
plt.tight_layout()
plt.show()


## 5. Clean & handle missing values
Both APIs return gap-free hourly series for this window (ERA5/CAMS are spatially and temporally complete), so we
expect near-zero NaNs. We still apply a defensive short-gap interpolation (limit=3 hours) and flag anything longer
as a real data-quality issue rather than silently filling it.


In [ ]:
def clean_series(df, cols, max_interp_gap=3):
    df = df.sort_values("timestamp").reset_index(drop=True)
    before = df[cols].isna().sum().sum()
    df[cols] = df[cols].interpolate(method="linear", limit=max_interp_gap, limit_direction="both")
    after = df[cols].isna().sum().sum()
    print(f"NaNs before interpolation: {before}, after: {after}")
    remaining = df[df[cols].isna().any(axis=1)]
    if len(remaining):
        print(f"WARNING: {len(remaining)} rows still have gaps longer than {max_interp_gap}h — inspect these, do not silently drop without checking.")
    return df

weather_cols = ["temperature", "humidity", "pressure", "rainfall_1h", "rain", "wind_speed", "wind_direction"]
aq_cols = ["PM2_5", "PM10", "NO2", "SO2", "O3", "AQI"]

weather_df = clean_series(weather_df, weather_cols)
aq_df = clean_series(aq_df, aq_cols)


## 6. Resample / align timestamps
Both sources are already hourly on the hour, so we align on an explicit hourly index and merge on `timestamp`.

In [ ]:
full_index = pd.DataFrame({
    "timestamp": pd.date_range(start=weather_df["timestamp"].min(),
                                end=weather_df["timestamp"].max(),
                                freq="h")
})

weather_aligned = full_index.merge(weather_df, on="timestamp", how="left")
aq_aligned = full_index.merge(aq_df, on="timestamp", how="left")

master = weather_aligned.merge(aq_aligned.drop(columns=["aqi_source"]).assign(aqi_source=aq_aligned["aqi_source"]),
                                on="timestamp", how="inner")
print("Master dataset shape:", master.shape)
master.head()


## 7. "Extract Chennai" / zone tagging
For M1 we use a single representative point (central Chennai). We tag it now so the schema already matches the
multi-zone master table planned for later milestones.


In [ ]:
master["zone_id"] = "CHN_CENTRAL"
master["latitude"] = CHENNAI_LAT
master["longitude"] = CHENNAI_LON
master = master[["timestamp", "zone_id", "latitude", "longitude"] +
                 weather_cols + aq_cols + ["aqi_source"]]
master.head()


## 8. Feature engineering — time, lag, rolling, rainfall accumulation
All lag/rolling features are computed strictly on **past** values only (no leakage from the future).


In [ ]:
df = master.copy()
df = df.sort_values("timestamp").reset_index(drop=True)

# --- Calendar features ---
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["month"] = df["timestamp"].dt.month
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

# --- Lag features (AQI and key pollutants/weather) ---
lag_hours = [1, 3, 6, 12, 24]
lag_cols = ["AQI", "PM2_5", "PM10", "temperature", "humidity", "wind_speed"]
for col in lag_cols:
    for lag in lag_hours:
        df[f"{col}_lag{lag}h"] = df[col].shift(lag)

# --- Rolling mean/std features (past window only, via shift(1) before rolling) ---
roll_windows = [3, 6, 12, 24]
for col in ["AQI", "PM2_5", "temperature", "humidity"]:
    for w in roll_windows:
        df[f"{col}_rollmean{w}h"] = df[col].shift(1).rolling(window=w, min_periods=max(2, w // 2)).mean()
        df[f"{col}_rollstd{w}h"] = df[col].shift(1).rolling(window=w, min_periods=max(2, w // 2)).std()

# --- Rainfall accumulation ---
for w in [1, 3, 6, 12, 24]:
    df[f"rainfall_{w}h"] = df["rainfall_1h"].shift(1).rolling(window=w, min_periods=1).sum()

print("Feature columns added. New shape:", df.shape)
df.head()


## 9. Prediction targets — AQI at +6h / +12h / +24h


In [ ]:
for h in [6, 12, 24]:
    df[f"AQI_target_{h}h"] = df["AQI"].shift(-h)

# Drop rows where we don't have enough lag history or enough future horizon
feature_cols = [c for c in df.columns if any(k in c for k in ["lag", "roll", "rainfall_", "hour", "day_of_week", "month", "is_weekend"])]
target_cols = ["AQI_target_6h", "AQI_target_12h", "AQI_target_24h"]

model_df = df.dropna(subset=feature_cols + target_cols).reset_index(drop=True)
print("Rows usable for modeling:", len(model_df), "out of", len(df))
model_df[["timestamp"] + feature_cols[:5] + target_cols].head()


## 10. Chronological train / validation / test split
No random shuffling — this is a time series. Roughly 70% / 15% / 15% by time order.


In [ ]:
n = len(model_df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = model_df.iloc[:train_end]
val_df = model_df.iloc[train_end:val_end]
test_df = model_df.iloc[val_end:]

print(f"Train: {len(train_df)} rows  ({train_df['timestamp'].min()} → {train_df['timestamp'].max()})")
print(f"Val:   {len(val_df)} rows  ({val_df['timestamp'].min()} → {val_df['timestamp'].max()})")
print(f"Test:  {len(test_df)} rows  ({test_df['timestamp'].min()} → {test_df['timestamp'].max()})")

X_train, X_val, X_test = train_df[feature_cols], val_df[feature_cols], test_df[feature_cols]


## 11. Baseline model — persistence
The simplest honest baseline: "AQI in H hours = AQI right now." Every real model must beat this.


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate(y_true, y_pred, label):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.clip(y_true, 1e-6, None))) * 100
    print(f"[{label}] MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.3f}  MAPE={mape:.1f}%")
    return {"MAE": mae, "RMSE": rmse, "R2": r2, "MAPE": mape}

results = {}
for h in [6, 12, 24]:
    y_true = test_df[f"AQI_target_{h}h"]
    y_pred_persistence = test_df["AQI"]  # "current AQI" as the forecast
    results[f"persistence_{h}h"] = evaluate(y_true, y_pred_persistence, f"Persistence +{h}h")


## 12. XGBoost model — one per horizon

In [ ]:
from xgboost import XGBRegressor

models = {}
for h in [6, 12, 24]:
    y_train = train_df[f"AQI_target_{h}h"]
    y_val = val_df[f"AQI_target_{h}h"]

    model = XGBRegressor(
        n_estimators=400,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="mae",
        early_stopping_rounds=30,
    )
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    models[h] = model
    print(f"Trained XGBoost for +{h}h — best iteration: {model.best_iteration}")


## 13. Evaluation — MAE / RMSE / R² / MAPE on the held-out test set

In [ ]:
for h in [6, 12, 24]:
    y_true = test_df[f"AQI_target_{h}h"]
    y_pred = models[h].predict(X_test)
    results[f"xgboost_{h}h"] = evaluate(y_true, y_pred, f"XGBoost +{h}h")

results_df = pd.DataFrame(results).T
results_df


## 14. Actual vs predicted — visual check

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
for ax, h in zip(axes, [6, 12, 24]):
    y_true = test_df[f"AQI_target_{h}h"].values
    y_pred = models[h].predict(X_test)
    ax.plot(test_df["timestamp"], y_true, label="Actual", linewidth=0.8)
    ax.plot(test_df["timestamp"], y_pred, label="Predicted (XGBoost)", linewidth=0.8, alpha=0.8)
    ax.set_title(f"AQI +{h}h — Actual vs Predicted")
    ax.legend()
plt.tight_layout()
plt.show()


## 15. Feature importance & SHAP explanation (for the +6h model)

In [ ]:
import shap

h = 6
explainer = shap.TreeExplainer(models[h])
shap_values = explainer(X_test)

shap.summary_plot(shap_values, X_test, max_display=15, show=True)


In [ ]:
# Native XGBoost feature importance (gain-based) for comparison
importance = pd.Series(models[6].feature_importances_, index=feature_cols).sort_values(ascending=False)
importance.head(15).plot(kind="barh", figsize=(8, 6), title="Top 15 features — XGBoost gain importance (+6h model)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Explain a single prediction (most recent test-set row) — this is the pattern the /api/explain endpoint will use later
row_idx = -1
single_row = X_test.iloc[[row_idx]]
single_shap = explainer(single_row)

print("Actual AQI (+6h):", test_df[f'AQI_target_6h'].iloc[row_idx])
print("Predicted AQI (+6h):", models[6].predict(single_row)[0])
shap.plots.waterfall(single_shap[0], show=True)


## 16. Save the final model and feature metadata

In [ ]:
import joblib
import json
import os

os.makedirs("models", exist_ok=True)

for h in [6, 12, 24]:
    joblib.dump(models[h], f"models/aqi_xgboost_{h}h.joblib")

metadata = {
    "feature_columns": feature_cols,
    "target_columns": target_cols,
    "zone_id": "CHN_CENTRAL",
    "latitude": CHENNAI_LAT,
    "longitude": CHENNAI_LON,
    "train_range": [str(train_df['timestamp'].min()), str(train_df['timestamp'].max())],
    "val_range": [str(val_df['timestamp'].min()), str(val_df['timestamp'].max())],
    "test_range": [str(test_df['timestamp'].min()), str(test_df['timestamp'].max())],
    "aqi_data_source": "openmeteo_reanalysis (CAMS) — NOT raw CPCB ground readings; see notebook intro",
    "weather_data_source": "openmeteo_archive (ERA5)",
    "test_metrics": {k: v for k, v in results.items() if k.startswith("xgboost")},
}

with open("models/model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2, default=str)

print("Saved 3 models + metadata to ./models/")
print(json.dumps(metadata, indent=2, default=str))


## Optional — start logging real CPCB ground-truth data (WAQI)
Run this once you have a free token from `https://aqicn.org/data-platform/token/`. It does **not** give you bulk
history, but it lets you start capturing real CPCB readings for Chennai stations *going forward* — which you'll
want for validating/calibrating the reanalysis-trained model above, and for the flood/incident milestones (M2+)
that benefit from real station-level granularity.

This cell is intentionally not run automatically — it needs your own token.


In [ ]:
WAQI_TOKEN = "PASTE_YOUR_TOKEN_HERE"  # get one free at https://aqicn.org/data-platform/token/

CHENNAI_STATIONS = [
    "manali", "alandur", "arumbakkam", "kodungaiyur",
    "perungudi", "royapuram", "velachery",
]

def fetch_waqi_station(station_name, token):
    url = f"https://api.waqi.info/feed/chennai/{station_name}/"
    r = requests.get(url, params={"token": token}, timeout=30)
    r.raise_for_status()
    return r.json()

if WAQI_TOKEN != "PASTE_YOUR_TOKEN_HERE":
    for station in CHENNAI_STATIONS:
        try:
            data = fetch_waqi_station(station, WAQI_TOKEN)
            print(station, "->", data.get("status"), data.get("data", {}).get("aqi"))
        except Exception as e:
            print(station, "-> ERROR:", e)
else:
    print("Set WAQI_TOKEN above to start pulling real CPCB station readings.")
